### Imports

In [ ]:
from qarp.optimizers import (
    ScipyOptimizer,
)
from qarp.optimizers import (
    SGDOptimizer,
    SPSAOptimizer,
    RMSPropOptimizer,
    AdamOptimizer,
    AdaGradOptimizer,
    AdamaxOptimizer,
    NadamOptimizer,
)

import numpy as np

In [ ]:
def optimize_with_callback(fun, grad, optimizer, n=2, x0=None, verbose=False):
    history = []
    if x0 is None:
        x0 = np.zeros(n) + 0.01

    def callback_with_value(xk):
        fk = fun(xk)
        evals = getattr(optimizer, "_nfev", None)
        history.append((xk.copy(), fk, evals))
        if verbose:
            print(f"Function value {fk}")

    uses_own_grad = (
        hasattr(optimizer, "options")
        and optimizer.options.get("grad_method") in {"spsa", "fd"}
    )
    effective_grad = None if uses_own_grad else grad

    result = optimizer.minimize(
        objective_function=fun,
        initial_parameters=x0,
        callback=callback_with_value,
        gradient=effective_grad,
    )
    return result, history


def run_optimizers(fun, grad, optimizers, names=None, n=2, x0=None, verbose=False):
    if names is None:
        names = [opt.__class__.__name__ for opt in optimizers]
    results = {}
    histories = {}
    for name, opt in zip(names, optimizers):
        result, history = optimize_with_callback(fun, grad, opt, n=n, x0=x0, verbose=verbose)
        results[name] = result
        histories[name] = history
        print(f"{name} optimized parameters: {result.x}")
    return results, histories

In [ ]:
def rosenbrock_function(x):
    return np.sum(100 * (x[1:] - x[:-1] ** 2) ** 2 + (1 - x[:-1]) ** 2)


def rosenbrock_gradient(x):
    jac = np.zeros_like(x)
    jac[:-1] += -400 * x[:-1] * (x[1:] - x[:-1] ** 2) - 2 * (1 - x[:-1])
    jac[1:] += 200 * (x[1:] - x[:-1] ** 2)


    return jac


In [ ]:
maxiter = 100
optimizers = [
    ScipyOptimizer(
        method="BFGS",
        options={
            "disp": False,
            "maxiter": maxiter,
        },
    ),
    SGDOptimizer(options={"lr": 0.005, "maxiter": maxiter}),
    SPSAOptimizer(
        options={
            "spsa_a0": 0.03,
            "spsa_alpha": 0.602,
            "spsa_A": 5.0,
            "spsa_c0": 0.05,
            "spsa_gamma": 0.101,
            "spsa_seed": 123,
            "grad_method": "spsa",
            "num_spsa": 5,
            "maxiter": maxiter,
        }
    ),
    RMSPropOptimizer(
        options={
            "lr": 0.01,
            "beta": 0.95,
            "maxiter": maxiter,
        }
    ),
    AdaGradOptimizer(options={"lr": 0.1, "maxiter": maxiter}),
    AdamOptimizer(options={"lr": 0.05, "maxiter": maxiter}),
    NadamOptimizer(options={"lr": 0.03, "maxiter": maxiter}),
    AdamaxOptimizer(options={"lr": 0.05, "maxiter": maxiter, "eps": 1e-8}),
]

function_to_opt = rosenbrock_function
function_gradient = rosenbrock_gradient

results, histories_rosenbrock = run_optimizers(
    function_to_opt, function_gradient, optimizers, verbose=False
)

In [ ]:
import matplotlib.pyplot as plt

for method_name, history in histories_rosenbrock.items():
    losses = [entry[1] for entry in history]
    plt.plot(np.arange(len(losses)), losses, label=method_name)
    
plt.title("Optimization of Rosenbrock function")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Another function.

Optimize another function that Scipy Optimizers will struggle with.

In [ ]:
def other_function(x):
    x0, x1 = x
    return 0.01 * x0**2 + x1**2 + 0.1 * np.sin(20 * x0)


def grad_other_function(x):
    x0, x1 = x
    jac = np.array([0.02 * x0 + 2.0 * np.cos(20 * x0), 2.0 * x1])
    return jac

In [ ]:
maxiter = 20
optimizers = [
    ScipyOptimizer(
        method="BFGS",
        options={
            "disp": False,
            "maxiter": maxiter,
        },
    ),
    SGDOptimizer(options={"lr": 0.005, "maxiter": maxiter}),
    SPSAOptimizer(
        options={
            "spsa_a0": 0.01,
            "spsa_alpha": 0.602,
            "spsa_A": 10.0,
            "spsa_c0": 0.1,
            "spsa_gamma": 0.101,
            "spsa_seed": 123,
            "grad_method": "spsa",
            "maxiter": maxiter,
        }
    ),
    RMSPropOptimizer(
        options={
            "lr": 0.01,
            "beta": 0.95,
            "maxiter": maxiter,
        }
    ),
    AdaGradOptimizer(options={"lr": 0.01, "maxiter": maxiter}),
    AdamOptimizer(options={"lr": 0.005, "maxiter": maxiter}),
    NadamOptimizer(options={"lr": 0.01, "maxiter": maxiter}),
    AdamaxOptimizer(options={"lr": 0.01, "maxiter": maxiter, "eps": 1e-8}),
]

In [ ]:
function_to_opt = other_function
function_gradient = grad_other_function

results, histories_other = run_optimizers(
    function_to_opt, function_gradient, optimizers, verbose=False
)

In [ ]:
import matplotlib.pyplot as plt

for method_name, history in histories_other.items():
    losses = [entry[1] for entry in history]
    plt.plot(np.arange(len(losses)), losses, label=method_name)

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Early Stopping

In [ ]:
maxiter = 300

# Early Stopping - when loss does not improve over 10 conscutive iterations by at least 1e-4
nadam = NadamOptimizer(options={"lr": 0.001, "maxiter": maxiter})
nadam_early_stop = NadamOptimizer(options={"lr": 0.001, "maxiter": maxiter, 
                                           "early_stopping" :True, "patience":10, "es_tol":1e-4})

results_early_stop, histories_early_stop = run_optimizers(
    function_to_opt, function_gradient, [nadam, nadam_early_stop], names=["nadam", "nadam_early_stop"],
      verbose=False
)

for method_name, history in histories_early_stop.items():
    losses = np.array([entry[1] for entry in history])
    plt.plot(np.arange(len(losses)), losses, label=method_name)

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()